# 🔍 Local RAG — PDF → ChromaDB → gpt-oss-20b via HuggingFace

**Stack:**
- 📄 PDF parsing → `pypdf`
- 🧠 Embeddings → Gemini `gemini-embedding-001` (free API)
- 🗄️ Vector DB → `ChromaDB` (local, persistent)
- 💬 LLM → `gpt-oss-20b` via HuggingFace Inference Providers (Groq)
- 🧠 Memory → Conversation checkpointing to disk
- 🔄 CDC → Chunk-level Change Data Capture for smart re-ingestion
- ⏱️ Recency → Exponential decay re-ranking on retrieval

---

#### ── Core Pipeline ──
- Cell 1  — Install dependencies
- Cell 2  — Config & API keys
- Cell 3  — PDF extraction
- Cell 4  — Chunking
- Cell 5  — Gemini batch embedding
- Cell 6  — ChromaDB storage & retrieval
- Cell 7  — ▶️ Run ingestion (point to your PDF here)
- Cell 8  — gpt-oss-20b answer generation (HuggingFace)
- Cell 9  — Full RAG pipeline — `ask()`
- Cell 10 — ▶️ Ask a single question
- Cell 11 — ▶️ Interactive REPL loop
- Cell 12 — ▶️ Utilities (stats, list docs, wipe DB)

#### ── ChromaDB Explorer ──
- Cell 13  — ▶️ Direct ChromaDB query explorer

#### ── Conversation Memory ──
- Cell 13A — Memory Manager (`ConversationMemory` class)
- Cell 13B — Memory-aware `ask_with_memory()`
- Cell 13C — ▶️ Interactive memory REPL
- Cell 13D — ▶️ Initialize memory session (run once per session)
- Cell 13E — ▶️ Ask a single question with memory
- Cell 13F — ▶️ Ask multiple questions with memory (follow-ups)

#### ── Staleness · CDC · Recency ──
- Cell 14A — Staleness Registry + enhanced `store_in_chroma_v2()`
- Cell 14B — CDC Engine + smart `ingest_pdf_v2()` entry point
- Cell 14C — Recency-weighted retrieval + `ask_weighted()`
- Cell 14D — ▶️ Test PDF generator (creates v1 / v2 / v3)
- Cell 14E — ▶️ Full test suite (staleness · CDC · recency · memory)

---

**Feature summary:**

| Feature | Cell | Description |
|---|---|---|
| Standard RAG | 1–11 | Ingest PDF → embed → retrieve → answer |
| DB Explorer | 13 | Peek, filter, keyword search, export ChromaDB |
| Memory | 13A–13F | Conversation history persisted to disk per session |
| Staleness | 14A | SHA256 file hash detects changed PDFs automatically |
| CDC | 14B | Only changed chunks re-embedded on document update |
| Recency | 14C | Newer chunks ranked higher via exponential decay |
| Testing | 14D–14E | Multi-version PDFs + end-to-end test suite |

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
# Run once. Restart kernel after installation completes.

# %pip install google-genai pypdf chromadb rich python-dotenv huggingface_hub fpdf2

In [1]:
# ── Cell 2: Config & API Keys ─────────────────────────────────────────────────
#
# Option A (recommended): Load from .env file in the same folder as this notebook
#   Create a .env file with:
#     GEMINI_API_KEY=your_gemini_key_here
#     HF_API_KEY=your_huggingface_token_here
#
# Option B: Paste keys directly below (never commit this to git)

import os
from dotenv import load_dotenv

load_dotenv()  # loads .env if present

# ── Fallback: paste directly if not using .env ────────────────────────────────
# os.environ["GEMINI_API_KEY"] = "paste_here"
# os.environ["HF_API_KEY"]     = "paste_here"

GEMINI_API_KEY     = os.environ.get("GEMINI_API_KEY", "")
EMBED_DIM          = 768
HF_API_KEY         = os.environ.get("HF_API_KEY", "")

# ── Model config ──────────────────────────────────────────────────────────────
GEMINI_EMBED_MODEL = "gemini-embedding-001"          # 768-dim, asymmetric retrieval
HF_LLM_MODEL       = "openai/gpt-oss-20b:groq"       # Use 20b for speed, swap to 120b for harder questions

# ── ChromaDB config ───────────────────────────────────────────────────────────
CHROMA_DB_PATH     = "./chroma_db"
COLLECTION_NAME    = "local_rag"

# ── Chunking config ───────────────────────────────────────────────────────────
CHUNK_SIZE         = 800    # characters per chunk
CHUNK_OVERLAP      = 120    # overlap between chunks

# ── Retrieval config ──────────────────────────────────────────────────────────
TOP_K              = 5      # number of chunks to retrieve per query

# ── Embedding batch config ────────────────────────────────────────────────────
EMBED_BATCH_SIZE   = 50     # chunks per Gemini API call
BATCH_SLEEP_SEC    = 0.3    # pause between batches

# ── LLM generation config ────────────────────────────────────────────────────
LLM_MAX_NEW_TOKENS = 1024
LLM_TEMPERATURE    = 0.1    # low = factual and grounded

# ── Validate keys ─────────────────────────────────────────────────────────────
assert GEMINI_API_KEY, "❌ GEMINI_API_KEY not set. Add it to .env or paste above."
assert HF_API_KEY,     "❌ HF_API_KEY not set. Add it to .env or paste above."

print("✅ Config loaded")
print(f"   Embedding model : {GEMINI_EMBED_MODEL}")
print(f"   LLM model       : {HF_LLM_MODEL}")
print(f"   ChromaDB path   : {CHROMA_DB_PATH}")
print(f"   Chunk size      : {CHUNK_SIZE} chars | Overlap: {CHUNK_OVERLAP} chars")

✅ Config loaded
   Embedding model : gemini-embedding-001
   LLM model       : openai/gpt-oss-20b:groq
   ChromaDB path   : ./chroma_db
   Chunk size      : 800 chars | Overlap: 120 chars


In [2]:
# ── Cell 3: PDF Text Extraction ───────────────────────────────────────────────

from pypdf import PdfReader

def extract_text_from_pdf(pdf_path: str) -> tuple[str, int]:
    """
    Extracts full text from a PDF file.
    Tags each page so chunks stay traceable back to their page number.
    Returns: (full_text, page_count)
    """
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            pages.append(f"[Page {i + 1}]\n{text.strip()}")
    return "\n\n".join(pages), len(reader.pages)

print("✅ extract_text_from_pdf() defined")

✅ extract_text_from_pdf() defined


In [3]:
# ── Cell 4: Text Chunking ─────────────────────────────────────────────────────

def chunk_text(text: str) -> list[str]:
    """
    Sliding window chunker with overlap.
    Drops fragments under 80 chars (stray headers, page numbers, etc.).
    """
    chunks, start = [], 0
    while start < len(text):
        end   = start + CHUNK_SIZE
        chunk = text[start:end].strip()
        if len(chunk) >= 80:
            chunks.append(chunk)
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks

print("✅ chunk_text() defined")

✅ chunk_text() defined


In [4]:
# ── Cell 5: Gemini Embedding (gemini-embedding-001 — one chunk at a time) ─────

import time
from google import genai
from google.genai import types

genai_client = genai.Client(api_key=GEMINI_API_KEY)


def embed_documents_batch(chunks: list[str]) -> list[list[float]]:
    """
    gemini-embedding-001 does NOT support batch input — one chunk per request.
    task_type='RETRIEVAL_DOCUMENT' for stored passages.
    output_dimensionality=768 truncates from 3072 — saves ChromaDB storage
    with minimal quality loss (MRL model).
    """
    all_embeddings = []
    total = len(chunks)

    for i, chunk in enumerate(chunks, 1):
        print(f"  Embedding chunk {i}/{total}...", end="\r")
        try:
            result = genai_client.models.embed_content(
                model=GEMINI_EMBED_MODEL,
                contents=chunk,
                config=types.EmbedContentConfig(
                    task_type="RETRIEVAL_DOCUMENT",
                    output_dimensionality=EMBED_DIM,
                ),
            )
            all_embeddings.append(result.embeddings[0].values)
            time.sleep(0.05)   # gentle rate-limit buffer

        except Exception as e:
            print(f"\n  ❌ Chunk {i} skipped: {e}")
            all_embeddings.append([0.0] * EMBED_DIM)  # placeholder keeps index aligned

    print(f"\n  ✅ Embedded {len(all_embeddings)} chunks")
    return all_embeddings


def embed_query(text: str) -> list[float]:
    """
    task_type='RETRIEVAL_QUERY' — asymmetric pair with RETRIEVAL_DOCUMENT.
    Must use same output_dimensionality as documents.
    """
    result = genai_client.models.embed_content(
        model=GEMINI_EMBED_MODEL,
        contents=text,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY",
            output_dimensionality=EMBED_DIM,
        ),
    )
    return result.embeddings[0].values


print(f"✅ Embedding functions defined")
print(f"   Model: {GEMINI_EMBED_MODEL} | Dimensions: {EMBED_DIM}")

✅ Embedding functions defined
   Model: gemini-embedding-001 | Dimensions: 768


In [5]:
# ── Cell 6: ChromaDB Storage & Retrieval ──────────────────────────────────────

import uuid
import chromadb

def get_collection():
    """Returns (or creates) the ChromaDB collection with cosine similarity."""
    client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
    return client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )


def store_in_chroma(chunks: list[str], embeddings: list[list[float]], doc_name: str) -> int:
    """Stores embedded chunks into ChromaDB with source metadata."""
    collection = get_collection()
    ids        = [str(uuid.uuid4()) for _ in chunks]
    metadatas  = [{"source": doc_name, "chunk_index": i} for i in range(len(chunks))]

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=chunks,
        metadatas=metadatas,
    )
    return len(chunks)


def retrieve_context(query: str) -> list[dict]:
    """Embeds query and retrieves top-K most similar chunks from ChromaDB."""
    collection      = get_collection()
    query_embedding = embed_query(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=TOP_K,
        include=["documents", "metadatas", "distances"],
    )

    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        chunks.append({
            "text":        doc,
            "source":      meta.get("source", "unknown"),
            "chunk_index": meta.get("chunk_index", -1),
            "score":       round(1 - dist, 4),  # cosine similarity: 1.0 = identical
        })

    return sorted(chunks, key=lambda x: x["score"], reverse=True)


print("✅ store_in_chroma() and retrieve_context() defined")

✅ store_in_chroma() and retrieve_context() defined


In [6]:
# ── Cell 7: ▶️ RUN INGESTION ──────────────────────────────────────────────────
# Change PDF_PATH to your actual PDF file path.
# Re-run this cell to ingest additional PDFs — ChromaDB appends, does not overwrite.

import os

PDF_PATH = "./pdfs/attention.pdf"

# ── Validate file ──────────────────────────────────────────────────────────────
assert os.path.exists(PDF_PATH), f"❌ File not found: {PDF_PATH}"

doc_name = os.path.basename(PDF_PATH)
print(f"📄 Ingesting: {doc_name}")
print("-" * 50)

# ── Step 1: Extract ────────────────────────────────────────────────────────────
print("Step 1/3  Extracting text from PDF...")
raw_text, page_count = extract_text_from_pdf(PDF_PATH)
print(f"✅ {page_count} pages | {len(raw_text):,} characters")

# ── Step 2: Chunk ──────────────────────────────────────────────────────────────
print("\nStep 2/3  Chunking text...")
chunks = chunk_text(raw_text)
print(f"✅ {len(chunks)} chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")

# ── Step 3: Embed + Store ─────────────────────────────────────────────────────
print(f"\nStep 3/3  Embedding via {GEMINI_EMBED_MODEL}...")
embeddings = embed_documents_batch(chunks)
stored     = store_in_chroma(chunks, embeddings, doc_name)

print("-" * 50)
print(f"✅ Done! {stored} chunks stored in ChromaDB → {CHROMA_DB_PATH}")

📄 Ingesting: attention.pdf
--------------------------------------------------
Step 1/3  Extracting text from PDF...
✅ 15 pages | 39,784 characters

Step 2/3  Chunking text...
✅ 59 chunks (size=800, overlap=120)

Step 3/3  Embedding via gemini-embedding-001...
  Embedding chunk 59/59...
  ✅ Embedded 59 chunks
--------------------------------------------------
✅ Done! 59 chunks stored in ChromaDB → ./chroma_db


In [7]:
# ── Cell 8: gpt-oss via HF Inference Providers (Groq) ────────────────────────

import os
from huggingface_hub import InferenceClient

# InferenceClient with provider routing — no model specified at init time
hf_client = InferenceClient(
    api_key=HF_API_KEY,
)

SYSTEM_MSG = (
    "You are a precise, document-grounded assistant. "
    "Answer ONLY using the CONTEXT provided. "
    "If the context is insufficient, say: "
    "'The provided documents don't cover this sufficiently.' "
    "Never fabricate facts. Be concise but complete. "
    "Use bullet points for multi-part answers."
)


def build_messages(query: str, context_chunks: list[dict]) -> list[dict]:
    context_blocks = [
        f"[Source: {c['source']} | Chunk #{c['chunk_index']} | Relevance: {c['score']}]\n{c['text']}"
        for c in context_chunks
    ]
    context_str  = "\n\n---\n\n".join(context_blocks)
    user_content = (
        f"Use the following context to answer the question.\n\n"
        f"CONTEXT:\n{context_str}\n\n"
        f"QUESTION:\n{query}"
    )
    return [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user",   "content": user_content},
    ]


def generate_answer(messages: list[dict]) -> str:
    """
    Uses HF Inference Providers routing.
    Model name format: "openai/gpt-oss-20b:groq"
    The provider suffix (:groq) tells HF router which backend to use.
    Groq is currently the fastest free provider for gpt-oss.
    """
    completion = hf_client.chat.completions.create(
        model=HF_LLM_MODEL,
        messages=messages,
        max_tokens=LLM_MAX_NEW_TOKENS,
        temperature=LLM_TEMPERATURE,
    )
    return completion.choices[0].message.content.strip()


print(f"✅ gpt-oss generation ready via HF Inference Providers")
print(f"   Model    : {HF_LLM_MODEL}")
print(f"   Provider : Groq (via HF router)")

✅ gpt-oss generation ready via HF Inference Providers
   Model    : openai/gpt-oss-20b:groq
   Provider : Groq (via HF router)


In [8]:
# ── Cell 9: Full RAG Pipeline ─────────────────────────────────────────────────

def ask(query: str, show_chunks: bool = True):
    print(f"\n{'='*60}")
    print(f"❓ Question: {query}")
    print(f"{'='*60}")

    # — Retrieve —
    print("🔍 Retrieving relevant chunks from ChromaDB...")
    chunks = retrieve_context(query)

    if not chunks:
        print("❌ No chunks found. Run Cell 7 (ingestion) first.")
        return

    # — Show sources —
    if show_chunks:
        print(f"\n📚 Top {len(chunks)} retrieved chunks:")
        for i, c in enumerate(chunks, 1):
            filled = int(c["score"] * 30)
            bar    = "█" * filled + "░" * (30 - filled)
            print(f"  {i}. {c['source']} chunk #{c['chunk_index']}  [{bar}]  {c['score']}")

    # — Generate —
    print(f"\n💬 Generating answer via {HF_LLM_MODEL}...\n")
    messages = build_messages(query, chunks)

    try:
        answer = generate_answer(messages)
        print(f"{'─'*60}")
        print("✅ Answer:")
        print(f"{'─'*60}")
        print(answer)
        print(f"{'─'*60}")
        return answer

    except Exception as e:
        err = str(e)
        if "403" in err or "gated" in err.lower():
            print("❌ Model access denied. Make sure your HF token has accepted")
            print("   the Llama 3.1 license at:")
            print("   https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct")
        elif "503" in err or "loading" in err.lower():
            print("⏳ Model cold-starting. Wait ~30s and re-run.")
        elif "429" in err or "rate" in err.lower():
            print("⚠️  Rate limit hit. Wait ~60s and retry.")
        else:
            print(f"❌ LLM error: {e}")

print("✅ ask() pipeline defined — ready to query")

✅ ask() pipeline defined — ready to query


In [9]:
# ── Cell 10: ▶️ ASK A SINGLE QUESTION ─────────────────────────────────────────
# Change the question below and re-run this cell anytime.

ask("What is the BLEU score on WMT 2014 English-to-German?")


❓ Question: What is the BLEU score on WMT 2014 English-to-German?
🔍 Retrieving relevant chunks from ChromaDB...

📚 Top 5 retrieved chunks:
  1. attention.pdf chunk #34  [██████████████████████░░░░░░░░]  0.7335
  2. attention.pdf chunk #33  [█████████████████████░░░░░░░░░]  0.7247
  3. attention.pdf chunk #1  [█████████████████████░░░░░░░░░]  0.7076
  4. attention.pdf chunk #35  [████████████████████░░░░░░░░░░]  0.6916
  5. attention.pdf chunk #31  [████████████████████░░░░░░░░░░]  0.6851

💬 Generating answer via openai/gpt-oss-20b:groq...

────────────────────────────────────────────────────────────
✅ Answer:
────────────────────────────────────────────────────────────
- The Transformer achieves a BLEU score of **28.4** on the WMT 2014 English‑to‑German translation task.
────────────────────────────────────────────────────────────


'- The Transformer achieves a BLEU score of **28.4** on the WMT\u202f2014 English‑to‑German translation task.'

In [10]:
# ── Cell 11: ▶️ INTERACTIVE REPL LOOP ─────────────────────────────────────────
# Type questions in the input box. Type 'exit' or 'quit' to stop.
# Note: Uses input() — works in Jupyter Lab and VS Code notebooks.

print("🚀 Local RAG — Interactive Mode")
print(f"   Embeddings : {GEMINI_EMBED_MODEL}")
print(f"   LLM        : {HF_LLM_MODEL}")
print(f"   Vector DB  : ChromaDB @ {CHROMA_DB_PATH}")
print("   Type 'exit' or 'quit' to stop.\n")

while True:
    try:
        query = input("You: ").strip()
        if not query:
            continue
        if query.lower() in ("exit", "quit", "q"):
            print("👋 Goodbye!")
            break
        ask(query)
    except KeyboardInterrupt:
        print("\n👋 Interrupted. Goodbye!")
        break

🚀 Local RAG — Interactive Mode
   Embeddings : gemini-embedding-001
   LLM        : openai/gpt-oss-20b:groq
   Vector DB  : ChromaDB @ ./chroma_db
   Type 'exit' or 'quit' to stop.

👋 Goodbye!


In [11]:
# ── Cell 12: UTILITIES ────────────────────────────────────────────────────────
# Helpful one-off commands — run individually as needed.

import chromadb

# ── Check how many chunks are in ChromaDB ─────────────────────────────────────
def check_db_stats():
    client     = chromadb.PersistentClient(path=CHROMA_DB_PATH)
    collection = client.get_or_create_collection(COLLECTION_NAME)
    count      = collection.count()
    print(f"📊 ChromaDB stats")
    print(f"   Collection : {COLLECTION_NAME}")
    print(f"   Total chunks stored : {count}")


# ── List all ingested source documents ───────────────────────────────────────
def list_ingested_docs():
    client     = chromadb.PersistentClient(path=CHROMA_DB_PATH)
    collection = client.get_or_create_collection(COLLECTION_NAME)
    results    = collection.get(include=["metadatas"])
    sources    = sorted(set(m["source"] for m in results["metadatas"]))
    print(f"📁 Ingested documents ({len(sources)}):")
    for s in sources:
        print(f"   - {s}")


# ── Wipe ChromaDB completely (start fresh) ────────────────────────────────────
def wipe_db():
    confirm = input("⚠️  This deletes ALL stored chunks. Type 'yes' to confirm: ")
    if confirm.strip().lower() == "yes":
        client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
        client.delete_collection(COLLECTION_NAME)
        print("🗑️  Collection wiped. Re-run ingestion to repopulate.")
    else:
        print("Cancelled.")


# ── Run whichever you need ────────────────────────────────────────────────────
check_db_stats()
# list_ingested_docs()
# wipe_db()

📊 ChromaDB stats
   Collection : local_rag
   Total chunks stored : 59


In [12]:
# ── Cell 13: ChromaDB Direct Query Explorer ───────────────────────────────────
# Query ChromaDB directly — no LLM involved.
# Run individual sections as needed.

import chromadb
import json

client     = chromadb.PersistentClient(path=CHROMA_DB_PATH)
collection = client.get_or_create_collection(COLLECTION_NAME)

print(f"📦 Collection : {COLLECTION_NAME}")
print(f"📊 Total chunks: {collection.count()}")
print("-" * 60)


# ════════════════════════════════════════════════════════════
# 1. PEEK — view first N chunks (raw text, no embedding)
# ════════════════════════════════════════════════════════════

def peek(n: int = 5):
    """View the first N stored chunks with their metadata."""
    results = collection.get(
        limit=n,
        include=["documents", "metadatas"]
    )
    print(f"\n📄 First {n} chunks:\n{'─'*60}")
    for i, (doc, meta) in enumerate(zip(results["documents"], results["metadatas"]), 1):
        print(f"[{i}] Source: {meta['source']} | Chunk #{meta['chunk_index']}")
        print(f"     {doc[:200]}...")   # first 200 chars
        print()


# ════════════════════════════════════════════════════════════
# 2. FILTER BY SOURCE — get all chunks from a specific PDF
# ════════════════════════════════════════════════════════════

def get_chunks_by_source(source_name: str):
    """Retrieve all chunks from a specific ingested PDF."""
    results = collection.get(
        where={"source": source_name},   # exact filename match
        include=["documents", "metadatas"]
    )
    print(f"\n📁 Chunks from '{source_name}': {len(results['documents'])} found\n{'─'*60}")
    for doc, meta in zip(results["documents"], results["metadatas"]):
        print(f"  Chunk #{meta['chunk_index']:>3} | {doc[:150]}...")


# ════════════════════════════════════════════════════════════
# 3. GET SPECIFIC CHUNK BY INDEX
# ════════════════════════════════════════════════════════════

def get_chunk(source_name: str, chunk_index: int):
    """Retrieve and print a specific chunk by source + index."""
    results = collection.get(
        where={"$and": [
            {"source": {"$eq": source_name}},
            {"chunk_index": {"$eq": chunk_index}}
        ]},
        include=["documents", "metadatas"]
    )
    if not results["documents"]:
        print(f"❌ No chunk found: {source_name} #{chunk_index}")
        return
    doc  = results["documents"][0]
    meta = results["metadatas"][0]
    print(f"\n🔎 {meta['source']} | Chunk #{meta['chunk_index']}\n{'─'*60}")
    print(doc)


# ════════════════════════════════════════════════════════════
# 4. SEMANTIC SEARCH — top-K similar chunks to a raw query
#    (same as RAG retrieval but prints raw chunks, no LLM)
# ════════════════════════════════════════════════════════════

def semantic_search(query: str, top_k: int = TOP_K):
    """
    Embed query and find similar chunks in ChromaDB.
    Shows raw retrieved chunks with scores — useful for
    debugging retrieval quality without LLM involvement.
    """
    query_embedding = embed_query(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    print(f"\n🔍 Semantic search: '{query}'\n{'─'*60}")
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        score = round(1 - dist, 4)
        filled = int(score * 30)
        bar    = "█" * filled + "░" * (30 - filled)
        print(f"  [{bar}] {score} | {meta['source']} chunk #{meta['chunk_index']}")
        print(f"  {doc[:250]}...")
        print()


# ════════════════════════════════════════════════════════════
# 5. KEYWORD SEARCH — find chunks containing a word/phrase
#    (simple text match, no embeddings needed)
# ════════════════════════════════════════════════════════════

def keyword_search(keyword: str):
    """
    Scan all stored chunks for a keyword/phrase (case-insensitive).
    Useful for finding exactly where a term appears in the DB.
    """
    all_data = collection.get(include=["documents", "metadatas"])
    matches  = [
        (doc, meta)
        for doc, meta in zip(all_data["documents"], all_data["metadatas"])
        if keyword.lower() in doc.lower()
    ]
    print(f"\n🔎 Keyword: '{keyword}' — {len(matches)} chunk(s) matched\n{'─'*60}")
    for doc, meta in matches:
        idx = doc.lower().find(keyword.lower())
        snippet = doc[max(0, idx-80):idx+120]   # context window around keyword
        print(f"  {meta['source']} chunk #{meta['chunk_index']}:")
        print(f"  ...{snippet}...")
        print()


# ════════════════════════════════════════════════════════════
# 6. EXPORT — dump all chunks to a readable JSON file
# ════════════════════════════════════════════════════════════

def export_to_json(output_path: str = "./chroma_export.json"):
    """Export all stored chunks to a JSON file for external inspection."""
    all_data = collection.get(include=["documents", "metadatas"])
    records  = [
        {"chunk_index": m["chunk_index"], "source": m["source"], "text": d}
        for d, m in zip(all_data["documents"], all_data["metadatas"])
    ]
    records.sort(key=lambda x: (x["source"], x["chunk_index"]))
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)
    print(f"✅ Exported {len(records)} chunks → {output_path}")


# ════════════════════════════════════════════════════════════
# ▶️  CALL WHICHEVER YOU NEED — uncomment and run
# ════════════════════════════════════════════════════════════

peek(n=3)                                          # view first 3 chunks
# get_chunks_by_source("attention.pdf")            # all chunks from a PDF
# get_chunk("attention.pdf", chunk_index=6)        # exact chunk that scored 0.72
# semantic_search("multi-head attention")          # retrieval debug, no LLM
# keyword_search("BLEU")                           # find where BLEU is mentioned
# export_to_json()                                 # dump everything to JSON

📦 Collection : local_rag
📊 Total chunks: 59
------------------------------------------------------------

📄 First 3 chunks:
────────────────────────────────────────────────────────────
[1] Source: attention.pdf | Chunk #0
     [Page 1]
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All ...

[2] Source: attention.pdf | Chunk #1
     ent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple netw...

[3] Source: attention.pdf | Chunk #2
     -to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.8 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best ...



In [13]:
# ── Cell 13A: Conversation Memory Manager ─────────────────────────────────────
# Maintains conversation history in-memory AND persists it to disk.
# Survives kernel restarts — resume any past session by loading its checkpoint.

import json
import os
from datetime import datetime

# ── Config ────────────────────────────────────────────────────────────────────
MEMORY_DIR        = "./memory_checkpoints"   # folder where sessions are saved
MAX_HISTORY_TURNS = 6                        # last N Q&A pairs sent to LLM
                                             # keeps prompt size manageable

os.makedirs(MEMORY_DIR, exist_ok=True)


class ConversationMemory:
    """
    Manages a conversation session with:
    - In-memory history (list of {role, content} dicts)
    - Automatic disk checkpointing after every turn
    - Resume from any saved session
    """

    def __init__(self, session_id: str = None):
        # Generate a timestamped session ID if not resuming an existing one
        self.session_id = session_id or datetime.now().strftime("%Y%m%d_%H%M%S")
        self.filepath   = os.path.join(MEMORY_DIR, f"{self.session_id}.json")
        self.history    = []   # list of {"role": ..., "content": ..., "timestamp": ...}
        self.metadata   = {
            "session_id": self.session_id,
            "created_at": datetime.now().isoformat(),
            "model":      HF_LLM_MODEL,
            "embed":      GEMINI_EMBED_MODEL,
            "turns":      0,
        }

        # Auto-load if resuming an existing session
        if os.path.exists(self.filepath):
            self._load()
            print(f"📂 Resumed session: {self.session_id}")
            print(f"   {self.metadata['turns']} previous turns loaded")
        else:
            print(f"🆕 New session: {self.session_id}")
            self._save()   # create file immediately

    # ── Add a turn ─────────────────────────────────────────────────────────────
    def add_turn(self, question: str, answer: str, chunks: list[dict]):
        """Add a Q&A turn to memory and checkpoint to disk immediately."""
        self.history.append({
            "role":      "user",
            "content":   question,
            "timestamp": datetime.now().isoformat(),
        })
        self.history.append({
            "role":      "assistant",
            "content":   answer,
            "timestamp": datetime.now().isoformat(),
            "sources":   [f"{c['source']} chunk #{c['chunk_index']} ({c['score']})"
                          for c in chunks],
        })
        self.metadata["turns"]      += 1
        self.metadata["updated_at"]  = datetime.now().isoformat()
        self._save()

    # ── Build message list for LLM ─────────────────────────────────────────────
    def get_messages_with_history(
        self,
        query: str,
        context_chunks: list[dict],
        system_msg: str,
    ) -> list[dict]:
        """
        Builds the full message list:
          system → (last N turns of history) → current context + question

        History gives the LLM awareness of prior exchanges.
        Context window kept bounded by MAX_HISTORY_TURNS.
        """
        messages = [{"role": "system", "content": system_msg}]

        # Inject last N turns (trim oldest first to stay within token budget)
        recent = self.history[-(MAX_HISTORY_TURNS * 2):]   # *2 because user+assistant
        for turn in recent:
            messages.append({
                "role":    turn["role"],
                "content": turn["content"],
            })

        # Current turn: context + question
        context_str  = "\n\n---\n\n".join([
            f"[Source: {c['source']} | Chunk #{c['chunk_index']} | Relevance: {c['score']}]\n{c['text']}"
            for c in context_chunks
        ])
        user_content = (
            f"Use the following context to answer the question. "
            f"You may also use our previous conversation if relevant.\n\n"
            f"CONTEXT:\n{context_str}\n\n"
            f"QUESTION:\n{query}"
        )
        messages.append({"role": "user", "content": user_content})
        return messages

    # ── Disk ops ───────────────────────────────────────────────────────────────
    def _save(self):
        with open(self.filepath, "w", encoding="utf-8") as f:
            json.dump({"metadata": self.metadata, "history": self.history},
                      f, indent=2, ensure_ascii=False)

    def _load(self):
        with open(self.filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.history  = data.get("history", [])
        self.metadata = data.get("metadata", self.metadata)

    # ── Display ────────────────────────────────────────────────────────────────
    def show_history(self):
        """Print conversation history in a readable format."""
        if not self.history:
            print("📭 No conversation history yet.")
            return
        print(f"\n📜 Session: {self.session_id} | {self.metadata['turns']} turns\n{'─'*60}")
        for turn in self.history:
            role    = "🧑 You" if turn["role"] == "user" else "🤖 AI"
            ts      = turn["timestamp"][:19].replace("T", " ")
            content = turn["content"]
            print(f"{role} [{ts}]:")
            print(f"  {content[:300]}{'...' if len(content) > 300 else ''}")
            if turn["role"] == "assistant" and "sources" in turn:
                print(f"  📎 Sources: {', '.join(turn['sources'])}")
            print()

    def clear(self):
        """Wipe history for this session (keeps the file, resets content)."""
        self.history            = []
        self.metadata["turns"]  = 0
        self._save()
        print(f"🗑️  History cleared for session: {self.session_id}")

    def summary(self):
        print(f"\n📊 Session summary")
        print(f"   ID        : {self.session_id}")
        print(f"   Turns     : {self.metadata['turns']}")
        print(f"   Model     : {self.metadata['model']}")
        print(f"   Saved to  : {self.filepath}")
        print(f"   Max history sent to LLM: {MAX_HISTORY_TURNS} turns")


# ── List all saved sessions ────────────────────────────────────────────────────
def list_sessions():
    files = sorted([f for f in os.listdir(MEMORY_DIR) if f.endswith(".json")])
    if not files:
        print("📭 No saved sessions found.")
        return
    print(f"\n💾 Saved sessions in '{MEMORY_DIR}':\n{'─'*60}")
    for f in files:
        path = os.path.join(MEMORY_DIR, f)
        with open(path) as fp:
            data = json.load(fp)
        meta = data.get("metadata", {})
        print(f"  📁 {meta.get('session_id', f)}")
        print(f"     Turns   : {meta.get('turns', '?')}")
        print(f"     Created : {meta.get('created_at', '?')[:19].replace('T', ' ')}")
        print(f"     Updated : {meta.get('updated_at', 'N/A')[:19].replace('T', ' ')}")
        print()


print("✅ ConversationMemory defined")
print(f"   Checkpoints saved to : {MEMORY_DIR}/")
print(f"   Max history per prompt: {MAX_HISTORY_TURNS} turns")

✅ ConversationMemory defined
   Checkpoints saved to : ./memory_checkpoints/
   Max history per prompt: 6 turns


In [14]:
# ── Cell 13B: Memory-aware ask() ──────────────────────────────────────────────
# Drop-in replacement for ask() — adds conversation memory on top.

SYSTEM_MSG_MEMORY = (
    "You are a precise, document-grounded assistant with memory of our conversation. "
    "Answer using ONLY the CONTEXT provided and prior conversation turns if relevant. "
    "If the context is insufficient, say: "
    "'The provided documents don't cover this sufficiently.' "
    "Never fabricate facts. Be concise but complete. "
    "Use bullet points for multi-part answers."
)


def ask_with_memory(query: str, memory: ConversationMemory, show_chunks: bool = True):
    """
    Full RAG pipeline with conversation memory:
      query → ChromaDB retrieval → history-aware prompt → LLM → checkpoint

    Args:
        query   : Your question
        memory  : ConversationMemory instance (carries session state)
        show_chunks: Print retrieved chunks (default True)
    """
    print(f"\n{'='*60}")
    print(f"❓ Question: {query}")
    print(f"   Session : {memory.session_id} | Turn #{memory.metadata['turns'] + 1}")
    print(f"{'='*60}")

    # — Retrieve —
    print("🔍 Retrieving relevant chunks from ChromaDB...")
    chunks = retrieve_context(query)

    if not chunks:
        print("❌ No chunks found. Run ingestion first.")
        return

    # — Show sources —
    if show_chunks:
        print(f"\n📚 Top {len(chunks)} retrieved chunks:")
        for i, c in enumerate(chunks, 1):
            filled = int(c["score"] * 30)
            bar    = "█" * filled + "░" * (30 - filled)
            print(f"  {i}. {c['source']} chunk #{c['chunk_index']}  [{bar}]  {c['score']}")

    # — Build history-aware messages —
    messages = memory.get_messages_with_history(query, chunks, SYSTEM_MSG_MEMORY)

    # — Generate —
    print(f"\n💬 Generating answer via {HF_LLM_MODEL}...\n")
    try:
        answer = generate_answer(messages)
        print(f"{'─'*60}")
        print("✅ Answer:")
        print(f"{'─'*60}")
        print(answer)
        print(f"{'─'*60}")

        # ── Checkpoint immediately after answer ────────────────────────────────
        memory.add_turn(question=query, answer=answer, chunks=chunks)
        print(f"💾 Checkpointed (turn #{memory.metadata['turns']})")

        return answer

    except Exception as e:
        err = str(e)
        if "503" in err or "loading" in err.lower():
            print("⏳ Model cold-starting. Wait ~30s and re-run.")
        elif "429" in err or "rate" in err.lower():
            print("⚠️  Rate limit hit. Wait ~60s and retry.")
        else:
            print(f"❌ LLM error: {e}")


print("✅ ask_with_memory() defined")

✅ ask_with_memory() defined


In [15]:
# ── Cell 13C: ▶️ Memory-aware REPL ────────────────────────────────────────────
#
# START A NEW SESSION:
#   memory = ConversationMemory()
#
# RESUME A PREVIOUS SESSION (paste session ID from list_sessions()):
#   memory = ConversationMemory("20240413_143022")
#
# VIEW ALL SAVED SESSIONS:
#   list_sessions()

# ── Uncomment ONE of these ─────────────────────────────────────────────────────
memory = ConversationMemory()                    # new session
# memory = ConversationMemory("paste_session_id_here")  # resume existing

print()
memory.summary()

# ── Interactive REPL with memory ──────────────────────────────────────────────
print("\n🚀 Memory-aware RAG — Interactive Mode")
print("   Commands: 'history' | 'summary' | 'clear' | 'sessions' | 'exit'\n")

while True:
    try:
        query = input("You: ").strip()
        if not query:
            continue
        if query.lower() in ("exit", "quit", "q"):
            print("👋 Goodbye! Session saved.")
            break
        elif query.lower() == "history":
            memory.show_history()
        elif query.lower() == "summary":
            memory.summary()
        elif query.lower() == "clear":
            memory.clear()
        elif query.lower() == "sessions":
            list_sessions()
        else:
            ask_with_memory(query, memory)
    except KeyboardInterrupt:
        print("\n👋 Interrupted. Session saved.")
        break

🆕 New session: 20260413_104959


📊 Session summary
   ID        : 20260413_104959
   Turns     : 0
   Model     : openai/gpt-oss-20b:groq
   Saved to  : ./memory_checkpoints\20260413_104959.json
   Max history sent to LLM: 6 turns

🚀 Memory-aware RAG — Interactive Mode
   Commands: 'history' | 'summary' | 'clear' | 'sessions' | 'exit'


❓ Question: What is the BLEU score on WMT 2014 English-to-German?
   Session : 20260413_104959 | Turn #1
🔍 Retrieving relevant chunks from ChromaDB...

📚 Top 5 retrieved chunks:
  1. attention.pdf chunk #34  [██████████████████████░░░░░░░░]  0.7335
  2. attention.pdf chunk #33  [█████████████████████░░░░░░░░░]  0.7247
  3. attention.pdf chunk #1  [█████████████████████░░░░░░░░░]  0.7076
  4. attention.pdf chunk #35  [████████████████████░░░░░░░░░░]  0.6916
  5. attention.pdf chunk #31  [████████████████████░░░░░░░░░░]  0.6851

💬 Generating answer via openai/gpt-oss-20b:groq...

────────────────────────────────────────────────────────────
✅ Answer:
─────

In [16]:
# ── Cell 13D: Initialize Memory Session ───────────────────────────────────────
# Run this ONCE per session before asking questions.
#
# New session:
memory = ConversationMemory()
#
# Resume a previous session (get ID from list_sessions()):
# memory = ConversationMemory("20240413_104959")

memory.summary()

🆕 New session: 20260413_105130

📊 Session summary
   ID        : 20260413_105130
   Turns     : 0
   Model     : openai/gpt-oss-20b:groq
   Saved to  : ./memory_checkpoints\20260413_105130.json
   Max history sent to LLM: 6 turns


In [17]:
# ── Cell 13E: ▶️ ASK A SINGLE QUESTION WITH MEMORY ───────────────────────────
# Change the question and re-run this cell anytime.
# Each run adds to the same session — follow-ups work naturally.

ask_with_memory("What is the BLEU score on WMT 2014 English-to-German?", memory)


❓ Question: What is the BLEU score on WMT 2014 English-to-German?
   Session : 20260413_105130 | Turn #1
🔍 Retrieving relevant chunks from ChromaDB...

📚 Top 5 retrieved chunks:
  1. attention.pdf chunk #34  [██████████████████████░░░░░░░░]  0.7335
  2. attention.pdf chunk #33  [█████████████████████░░░░░░░░░]  0.7247
  3. attention.pdf chunk #1  [█████████████████████░░░░░░░░░]  0.7076
  4. attention.pdf chunk #35  [████████████████████░░░░░░░░░░]  0.6916
  5. attention.pdf chunk #31  [████████████████████░░░░░░░░░░]  0.6851

💬 Generating answer via openai/gpt-oss-20b:groq...

────────────────────────────────────────────────────────────
✅ Answer:
────────────────────────────────────────────────────────────
- The Transformer (big) model achieves a BLEU score of **28.4** on the WMT 2014 English‑to‑German translation task.
────────────────────────────────────────────────────────────
💾 Checkpointed (turn #1)


'- The Transformer (big) model achieves a BLEU score of **28.4** on the WMT\u202f2014 English‑to‑German translation task.'

In [18]:
# ── Cell 13F: ▶️ ASK A MULTIPLE QUESTION WITH MEMORY ───────────────────────────
# Change the question and re-run this cell anytime.
# Each run adds to the same session — follow-ups work naturally.

ask_with_memory("What architecture does this paper propose?", memory)
ask_with_memory("How many attention heads does it use?", memory)
ask_with_memory("How does that compare to what you said about the BLEU score?", memory)  # ← follow-up works


❓ Question: What architecture does this paper propose?
   Session : 20260413_105130 | Turn #2
🔍 Retrieving relevant chunks from ChromaDB...

📚 Top 5 retrieved chunks:
  1. attention.pdf chunk #6  [█████████████████████░░░░░░░░░]  0.7243
  2. attention.pdf chunk #9  [█████████████████████░░░░░░░░░]  0.7182
  3. attention.pdf chunk #10  [█████████████████████░░░░░░░░░]  0.7162
  4. attention.pdf chunk #1  [█████████████████████░░░░░░░░░]  0.7152
  5. attention.pdf chunk #11  [█████████████████████░░░░░░░░░]  0.7079

💬 Generating answer via openai/gpt-oss-20b:groq...

────────────────────────────────────────────────────────────
✅ Answer:
────────────────────────────────────────────────────────────
- The paper introduces the **Transformer** architecture.  
- It is a sequence‑to‑sequence model that relies **entirely on attention mechanisms** (self‑attention and multi‑head attention).  
- The design eliminates recurrence and convolution, using only stacked encoder and decoder layers compose

'- The context states that the Transformer (big) model achieves a **new state‑of‑the‑art BLEU score of 28.4** on WMT\u202f2014 English‑to‑German.  \n- This matches exactly what I previously reported.  \n- Therefore, there is no discrepancy: the context confirms the BLEU score I gave.'

In [19]:
# ── Cell 14A: Content Staleness Tracking ──────────────────────────────────────
# Tracks per-document: file hash, ingest timestamp, version counter.
# Detects if a PDF on disk has changed since it was last ingested.
# Adds file_hash + ingested_at to every chunk's ChromaDB metadata.

import hashlib
import json
import uuid
import os
from datetime import datetime

STALENESS_REGISTRY_PATH = "./staleness_registry.json"
CHUNK_REGISTRY_PATH     = "./chunk_registry.json"


# ════════════════════════════════════════════════════════════
# 1. HASH UTILITIES
# ════════════════════════════════════════════════════════════

def compute_file_hash(pdf_path: str) -> str:
    """SHA256 of entire PDF binary — changes if any byte changes."""
    sha = hashlib.sha256()
    with open(pdf_path, "rb") as f:
        for block in iter(lambda: f.read(65536), b""):
            sha.update(block)
    return sha.hexdigest()


def compute_chunk_hash(text: str) -> str:
    """MD5 of chunk text — fast, good enough for CDC diffing."""
    return hashlib.md5(text.encode("utf-8")).hexdigest()


# ════════════════════════════════════════════════════════════
# 2. STALENESS REGISTRY
# ════════════════════════════════════════════════════════════

class StalenessRegistry:
    """
    Persists {doc_name: {file_hash, ingested_at, chunk_count, version}}
    to staleness_registry.json.
    """
    def __init__(self, path: str = STALENESS_REGISTRY_PATH):
        self.path = path
        self.data = {}
        if os.path.exists(path):
            with open(path) as f:
                self.data = json.load(f)

    def _save(self):
        with open(self.path, "w") as f:
            json.dump(self.data, f, indent=2)

    def register(self, doc_name: str, file_hash: str, chunk_count: int):
        """Record a successful ingestion."""
        prev_version = self.data.get(doc_name, {}).get("version", 0)
        self.data[doc_name] = {
            "file_hash":   file_hash,
            "chunk_count": chunk_count,
            "ingested_at": datetime.now().isoformat(),
            "version":     prev_version + 1,
        }
        self._save()

    def check(self, pdf_path: str) -> dict:
        """
        Returns dict with one of three statuses:
          'new'       — never seen this doc before
          'unchanged' — file hash matches stored hash
          'stale'     — file hash changed since last ingest
        """
        doc_name     = os.path.basename(pdf_path)
        current_hash = compute_file_hash(pdf_path)

        if doc_name not in self.data:
            return {"status": "new", "doc_name": doc_name,
                    "file_hash": current_hash}

        stored = self.data[doc_name]
        if stored["file_hash"] == current_hash:
            return {"status": "unchanged", "doc_name": doc_name,
                    "file_hash": current_hash,
                    "ingested_at": stored["ingested_at"],
                    "version": stored["version"]}

        return {"status": "stale", "doc_name": doc_name,
                "file_hash": current_hash,
                "old_hash": stored["file_hash"],
                "ingested_at": stored["ingested_at"],
                "version": stored["version"]}

    def show(self):
        if not self.data:
            print("📭 Staleness registry is empty.")
            return
        print(f"\n📋 Staleness Registry ({len(self.data)} docs)\n{'─'*60}")
        for doc, info in self.data.items():
            status_icon = "✅"
            print(f"  {status_icon} {doc}")
            print(f"     Version    : v{info['version']}")
            print(f"     Ingested   : {info['ingested_at'][:19].replace('T',' ')}")
            print(f"     Chunks     : {info['chunk_count']}")
            print(f"     Hash       : {info['file_hash'][:24]}...")
            print()


# ════════════════════════════════════════════════════════════
# 3. ENHANCED store_in_chroma
#    Adds file_hash + ingested_at to every chunk's metadata.
#    Returns {chunk_hash: chroma_id} for the CDC registry.
# ════════════════════════════════════════════════════════════

def store_in_chroma_v2(
    chunks:      list[str],
    embeddings:  list[list[float]],
    doc_name:    str,
    file_hash:   str,
    ingested_at: str,
) -> dict:
    """
    Enhanced ChromaDB store.
    Each chunk gets: source, chunk_index, file_hash, ingested_at.
    Returns {chunk_hash: chroma_id} — consumed by ChunkRegistry.
    """
    collection     = get_collection()
    ids            = [str(uuid.uuid4()) for _ in chunks]
    metadatas      = [
        {
            "source":      doc_name,
            "chunk_index": i,
            "file_hash":   file_hash,
            "ingested_at": ingested_at,
        }
        for i in range(len(chunks))
    ]

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=chunks,
        metadatas=metadatas,
    )

    # Build {chunk_hash: chroma_id} for CDC tracking
    return {compute_chunk_hash(chunk): chroma_id
            for chunk, chroma_id in zip(chunks, ids)}


# ── Initialise ────────────────────────────────────────────────────────────────
staleness_reg = StalenessRegistry()

print("✅ Cell 14A: Staleness Registry ready")
staleness_reg.show()
print(f"   Registry file : {STALENESS_REGISTRY_PATH}")
print(f"   Chunk reg file: {CHUNK_REGISTRY_PATH}")

✅ Cell 14A: Staleness Registry ready
📭 Staleness registry is empty.
   Registry file : ./staleness_registry.json
   Chunk reg file: ./chunk_registry.json


In [20]:
# ── Cell 14B: CDC Engine + Smart Ingest ───────────────────────────────────────
# Chunk-level Change Data Capture:
#   - Hashes every chunk text individually
#   - Diffs old vs new hashes on re-ingest
#   - Deletes only removed chunks from ChromaDB
#   - Embeds + inserts only added chunks
#   - Unchanged chunks are untouched (zero API calls wasted)

# ════════════════════════════════════════════════════════════
# 1. CHUNK REGISTRY
# ════════════════════════════════════════════════════════════

class ChunkRegistry:
    """
    Persists {doc_name: {chunk_hash: chroma_id}} to chunk_registry.json.
    The hash key enables O(1) lookup to check if a chunk already exists.
    """
    def __init__(self, path: str = CHUNK_REGISTRY_PATH):
        self.path = path
        self.data = {}
        if os.path.exists(path):
            with open(path) as f:
                self.data = json.load(f)

    def _save(self):
        with open(self.path, "w") as f:
            json.dump(self.data, f, indent=2)

    def get_doc(self, doc_name: str) -> dict:
        return self.data.get(doc_name, {})

    def update(self, doc_name: str, registry: dict):
        self.data[doc_name] = registry
        self._save()

    def show(self):
        if not self.data:
            print("📭 Chunk registry is empty.")
            return
        print(f"\n🗂️  Chunk Registry ({len(self.data)} docs):")
        for doc, reg in self.data.items():
            print(f"   {doc}: {len(reg)} chunks tracked")


# ════════════════════════════════════════════════════════════
# 2. DIFF ENGINE
# ════════════════════════════════════════════════════════════

def diff_chunks(old_registry: dict, new_chunks: list[str]) -> dict:
    """
    Compares old {chunk_hash: chroma_id} against new chunk texts.

    Returns:
      added:     {chunk_hash: chunk_text}   — needs embedding + insert
      removed:   {chunk_hash: chroma_id}    — needs delete from ChromaDB
      unchanged: {chunk_hash: chroma_id}    — kept as-is, zero API cost
    """
    new_hash_map = {compute_chunk_hash(c): c for c in new_chunks}
    old_hashes   = set(old_registry.keys())
    new_hashes   = set(new_hash_map.keys())

    return {
        "added":     {h: new_hash_map[h] for h in (new_hashes - old_hashes)},
        "removed":   {h: old_registry[h] for h in (old_hashes - new_hashes)},
        "unchanged": {h: old_registry[h] for h in (old_hashes & new_hashes)},
    }


# ════════════════════════════════════════════════════════════
# 3. CDC UPDATE
# ════════════════════════════════════════════════════════════

def cdc_update(pdf_path: str) -> dict:
    """
    Surgical CDC update for a changed PDF:
      1. Extract + chunk new version
      2. Diff chunk hashes vs stored registry
      3. Delete removed chunks from ChromaDB
      4. Embed + insert ONLY added chunks
      5. Update both staleness + chunk registries
    """
    doc_name    = os.path.basename(pdf_path)
    file_hash   = compute_file_hash(pdf_path)
    ingested_at = datetime.now().isoformat()

    # Step 1: Extract + chunk
    print("\n📄 Extracting new version...")
    raw_text, page_count = extract_text_from_pdf(pdf_path)
    new_chunks = chunk_text(raw_text)
    print(f"   {page_count} pages → {len(new_chunks)} chunks")

    # Step 2: Diff
    old_registry = chunk_reg.get_doc(doc_name)
    diff         = diff_chunks(old_registry, new_chunks)

    n_added     = len(diff["added"])
    n_removed   = len(diff["removed"])
    n_unchanged = len(diff["unchanged"])
    total       = n_unchanged + n_added

    print(f"\n📊 Chunk-level diff result:")
    print(f"   ✅ Unchanged : {n_unchanged:>4}  (kept — zero re-embedding cost)")
    print(f"   ➕ Added     : {n_added:>4}  (will be embedded + inserted)")
    print(f"   ➖ Removed   : {n_removed:>4}  (will be deleted from ChromaDB)")
    if total > 0:
        saved_pct = int(100 * n_unchanged / total)
        print(f"   💰 API calls saved: {n_unchanged}/{total} ({saved_pct}% reuse)")

    collection       = get_collection()
    new_doc_registry = dict(diff["unchanged"])  # start with unchanged

    # Step 3: Delete removed chunks
    if diff["removed"]:
        ids_to_delete = list(diff["removed"].values())
        collection.delete(ids=ids_to_delete)
        print(f"\n🗑️  Deleted {len(ids_to_delete)} stale chunks from ChromaDB")

    # Step 4: Embed + insert added chunks
    if diff["added"]:
        added_hashes = list(diff["added"].keys())
        added_texts  = list(diff["added"].values())

        print(f"\n🧠 Embedding {len(added_texts)} new/changed chunks...")
        new_embeddings = embed_documents_batch(added_texts)

        # Assign chunk indices sequentially after existing max
        existing      = collection.get(
            where={"source": doc_name}, include=["metadatas"]
        )
        existing_idxs = [m["chunk_index"] for m in (existing["metadatas"] or [])]
        next_idx      = max(existing_idxs, default=-1) + 1

        new_ids, new_metas = [], []
        for i, (h, text) in enumerate(zip(added_hashes, added_texts)):
            cid = str(uuid.uuid4())
            new_ids.append(cid)
            new_metas.append({
                "source":      doc_name,
                "chunk_index": next_idx + i,
                "file_hash":   file_hash,
                "ingested_at": ingested_at,   # ← fresh timestamp → high recency score
            })
            new_doc_registry[h] = cid

        collection.add(
            ids=new_ids,
            embeddings=new_embeddings,
            documents=added_texts,
            metadatas=new_metas,
        )
        print(f"✅ Inserted {len(new_ids)} new chunks")

    # Step 5: Update registries
    chunk_reg.update(doc_name, new_doc_registry)
    staleness_reg.register(doc_name, file_hash, len(new_doc_registry))

    print(f"\n💾 CDC complete — registries updated")
    return {"status": "updated", "added": n_added,
            "removed": n_removed, "unchanged": n_unchanged}


# ════════════════════════════════════════════════════════════
# 4. SMART INGEST (single entry point for all scenarios)
# ════════════════════════════════════════════════════════════

def ingest_pdf_v2(pdf_path: str, force: bool = False):
    """
    Smart ingest — automatically handles all three cases:
      'new'       → full ingest (extract, chunk, embed, store)
      'stale'     → CDC update (only changed chunks re-embedded)
      'unchanged' → skip (unless force=True)
    """
    doc_name = os.path.basename(pdf_path)
    print(f"\n{'='*60}")
    print(f"📥 Smart Ingest: {doc_name}")
    print(f"{'='*60}")

    status = staleness_reg.check(pdf_path)
    print(f"📌 Staleness status: {status['status'].upper()}")

    # ── Unchanged ──────────────────────────────────────────────────────────────
    if status["status"] == "unchanged" and not force:
        print(f"✅ Already up to date")
        print(f"   Version    : v{status['version']}")
        print(f"   Ingested   : {status['ingested_at'][:19]}")
        print("   Use force=True to re-ingest anyway.")
        return

    # ── Stale → route to CDC ───────────────────────────────────────────────────
    if status["status"] == "stale":
        v = status["version"]
        print(f"🔄 Changes detected (was v{v}) — running CDC update...")
        return cdc_update(pdf_path)

    # ── New (or force) → full ingest ───────────────────────────────────────────
    ingested_at = datetime.now().isoformat()
    file_hash   = status["file_hash"]

    print("📄 Step 1/3  Extracting text...")
    raw_text, page_count = extract_text_from_pdf(pdf_path)
    print(f"   ✅ {page_count} pages | {len(raw_text):,} chars")

    print("✂️  Step 2/3  Chunking...")
    chunks = chunk_text(raw_text)
    print(f"   ✅ {len(chunks)} chunks")

    print(f"🧠 Step 3/3  Embedding via {GEMINI_EMBED_MODEL}...")
    embeddings = embed_documents_batch(chunks)

    chunk_hash_map = store_in_chroma_v2(
        chunks, embeddings, doc_name, file_hash, ingested_at
    )
    chunk_reg.update(doc_name, chunk_hash_map)
    staleness_reg.register(doc_name, file_hash, len(chunks))

    print(f"\n✅ Full ingest complete!")
    print(f"   {len(chunks)} chunks stored")
    print(f"   Hash       : {file_hash[:24]}...")
    print(f"   Ingested at: {ingested_at[:19]}")


# ── Initialise ────────────────────────────────────────────────────────────────
chunk_reg = ChunkRegistry()

print("✅ Cell 14B: CDC Engine + Smart Ingest ready")
chunk_reg.show()

✅ Cell 14B: CDC Engine + Smart Ingest ready
📭 Chunk registry is empty.


In [21]:
# ── Cell 14C: Recency-Weighted Retrieval ──────────────────────────────────────
# Blends cosine similarity with an exponential recency decay score.
# Newer chunks (fresher ingested_at) rank slightly higher than older ones
# with equal semantic similarity — critical for living documents.

import math

# ── Tuning knobs ──────────────────────────────────────────────────────────────
RECENCY_ALPHA     = 0.85   # cosine weight  (1-alpha = recency weight)
RECENCY_HALF_LIFE = 30     # days — recency score halves every 30 days
#
# Tune alpha per use case:
#   Research papers (static)    → 0.95
#   Quarterly reports           → 0.85  ← default
#   Daily news / live docs      → 0.70


# ════════════════════════════════════════════════════════════
# 1. RECENCY DECAY FUNCTION
# ════════════════════════════════════════════════════════════

def recency_decay(ingested_at_str: str,
                  half_life_days: float = RECENCY_HALF_LIFE) -> float:
    """
    Exponential decay: score = e^(-λ * days_elapsed)
    where λ = ln(2) / half_life_days

    Score table (half_life=30 days):
      0 days ago  → 1.000  (brand new)
      7 days ago  → 0.857
      15 days ago → 0.707
      30 days ago → 0.500
      60 days ago → 0.250
    """
    try:
        ingested  = datetime.fromisoformat(ingested_at_str)
        days_gone = (datetime.now() - ingested).total_seconds() / 86400
        λ         = math.log(2) / half_life_days
        return round(math.exp(-λ * days_gone), 4)
    except Exception:
        return 0.5   # fallback: mid-range score for chunks without ingested_at


# ════════════════════════════════════════════════════════════
# 2. WEIGHTED RETRIEVAL
# ════════════════════════════════════════════════════════════

def retrieve_context_weighted(
    query:       str,
    alpha:       float = RECENCY_ALPHA,
    show_scores: bool  = False,
) -> list[dict]:
    """
    Retrieves TOP_K*3 candidates by cosine similarity,
    then re-ranks by blended score:
        final_score = alpha * cosine + (1 - alpha) * recency
    Returns top TOP_K after re-ranking.
    """
    collection      = get_collection()
    query_embedding = embed_query(query)

    # Fetch extra candidates so re-ranking has meaningful room to work
    n_candidates = min(TOP_K * 3, collection.count())

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_candidates,
        include=["documents", "metadatas", "distances"],
    )

    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        cosine_score  = round(1 - dist, 4)
        ingested_at   = meta.get("ingested_at", datetime.now().isoformat())
        recency_score = recency_decay(ingested_at)
        blended       = round(alpha * cosine_score + (1 - alpha) * recency_score, 4)

        chunks.append({
            "text":          doc,
            "source":        meta.get("source", "unknown"),
            "chunk_index":   meta.get("chunk_index", -1),
            "ingested_at":   ingested_at[:19].replace("T", " "),
            "cosine_score":  cosine_score,
            "recency_score": recency_score,
            "score":         blended,
        })

    chunks.sort(key=lambda x: x["score"], reverse=True)
    top_chunks = chunks[:TOP_K]

    if show_scores:
        print(f"\n📊 Score breakdown  (α={alpha} cosine | {round(1-alpha,2)} recency | "
              f"half-life={RECENCY_HALF_LIFE}d):")
        print(f"   {'#':>3}  {'chunk':>8}  {'cosine':>8}  {'recency':>8}  "
              f"{'blended':>8}  ingested")
        print(f"   {'─'*65}")
        for i, c in enumerate(top_chunks, 1):
            print(f"   {i:>3}  #{c['chunk_index']:>6}  "
                  f"{c['cosine_score']:>8}  "
                  f"{c['recency_score']:>8}  "
                  f"{c['score']:>8}  "
                  f"{c['ingested_at']}")

    return top_chunks


# ════════════════════════════════════════════════════════════
# 3. ask_weighted — drop-in replacement for ask() / ask_with_memory()
# ════════════════════════════════════════════════════════════

def ask_weighted(
    query:       str,
    memory:      object = None,    # pass ConversationMemory for history-aware mode
    alpha:       float  = RECENCY_ALPHA,
    show_chunks: bool   = True,
    show_scores: bool   = False,   # True = print cosine/recency breakdown
):
    """
    Full RAG pipeline with recency-weighted retrieval.
    Works standalone or with ConversationMemory.
    """
    print(f"\n{'='*60}")
    print(f"❓ {query}")
    if memory:
        print(f"   Session : {memory.session_id} | "
              f"Turn #{memory.metadata['turns'] + 1}")
    print(f"   Retrieval: α={alpha} cosine | {round(1-alpha,2)} recency")
    print(f"{'='*60}")

    # Weighted retrieval
    print("🔍 Retrieving with recency weighting...")
    chunks = retrieve_context_weighted(query, alpha=alpha, show_scores=show_scores)

    if not chunks:
        print("❌ No chunks found. Run ingest_pdf_v2() first.")
        return

    # Display
    if show_chunks:
        print(f"\n📚 Top {len(chunks)} chunks:")
        for i, c in enumerate(chunks, 1):
            filled = int(c["score"] * 30)
            bar    = "█" * filled + "░" * (30 - filled)
            print(
                f"  {i}. {c['source']} #{c['chunk_index']:>3} "
                f"[{bar}] {c['score']}"
                f"  (cos={c['cosine_score']}, rec={c['recency_score']},"
                f" 📅{c['ingested_at'][5:16]})"
            )

    # Build messages
    if memory:
        messages = memory.get_messages_with_history(
            query, chunks, SYSTEM_MSG_MEMORY
        )
    else:
        messages = build_messages(query, chunks)

    # Generate
    print(f"\n💬 Generating via {HF_LLM_MODEL}...\n")
    try:
        answer = generate_answer(messages)
        print(f"{'─'*60}")
        print("✅ Answer:")
        print(f"{'─'*60}")
        print(answer)
        print(f"{'─'*60}")

        if memory:
            memory.add_turn(query, answer, chunks)
            print(f"💾 Checkpointed (turn #{memory.metadata['turns']})")

        return answer

    except Exception as e:
        print(f"❌ LLM error: {e}")


print("✅ Cell 14C: Recency-Weighted Retrieval ready")
print(f"   Alpha (cosine weight)  : {RECENCY_ALPHA}")
print(f"   Recency half-life      : {RECENCY_HALF_LIFE} days")
print(f"   Candidates fetched     : TOP_K × 3 = {TOP_K * 3}")

✅ Cell 14C: Recency-Weighted Retrieval ready
   Alpha (cosine weight)  : 0.85
   Recency half-life      : 30 days
   Candidates fetched     : TOP_K × 3 = 15


In [23]:
# ── Cell 14D: Test PDF Generator ──────────────────────────────────────────────
# Creates 3 versions of "RAG Systems Guide" with controlled content changes.
# Designed to produce clear CDC diffs and recency score differences.
#
# Version layout:
#   v1: Introduction + Vector Databases (original) + Embedding Models
#   v2: Introduction (same) + Vector Databases (updated) + Embedding (same) + LLM (new)
#   v3: Introduction (same) + Vector Databases (same as v2) + Embedding (same)
#                           + LLM Integration (updated) + RAG Evaluation (new)

from fpdf import FPDF
import os, shutil

os.makedirs("./pdfs/versions", exist_ok=True)

# ── PDF builder ───────────────────────────────────────────────────────────────
def make_pdf(path: str, title: str, sections: dict):
    """
    Builds a PDF. Sanitizes text to latin-1 safe characters
    so fpdf2's built-in Helvetica font doesn't throw UnicodeEncodeError.
    """
    def sanitize(text: str) -> str:
        # Replace common Unicode punctuation with ASCII equivalents
        replacements = {
            "\u2014": "-",   # em dash —
            "\u2013": "-",   # en dash –
            "\u2018": "'",   # left single quote '
            "\u2019": "'",   # right single quote '
            "\u201c": '"',   # left double quote "
            "\u201d": '"',   # right double quote "
            "\u2026": "...", # ellipsis …
            "\u00a0": " ",   # non-breaking space
        }
        for char, replacement in replacements.items():
            text = text.replace(char, replacement)
        # Final fallback — drop anything still outside latin-1
        return text.encode("latin-1", errors="ignore").decode("latin-1")

    pdf = FPDF()
    pdf.set_margins(20, 20, 20)
    pdf.add_page()

    pdf.set_font("Helvetica", "B", 18)
    pdf.cell(0, 14, sanitize(title), new_x="LMARGIN", new_y="NEXT")
    pdf.ln(2)
    pdf.set_draw_color(100, 100, 100)
    pdf.line(20, pdf.get_y(), 190, pdf.get_y())
    pdf.ln(6)

    for heading, body in sections.items():
        pdf.set_font("Helvetica", "B", 13)
        pdf.cell(0, 10, sanitize(heading), new_x="LMARGIN", new_y="NEXT")
        pdf.set_font("Helvetica", size=11)
        pdf.multi_cell(0, 7, sanitize(body))
        pdf.ln(5)

    pdf.output(path)
    print(f"✅ Created: {path}")


# ════════════════════════════════════════════════════════════
# CONTENT BLOCKS
# ════════════════════════════════════════════════════════════

# ── Shared across all versions (tests that CDC correctly skips these) ──────────
INTRO = (
    "Retrieval-Augmented Generation (RAG) combines a retrieval system with a "
    "large language model. Instead of relying on parametric knowledge alone, "
    "RAG fetches relevant documents from an external knowledge base and injects "
    "them into the model's context window. This grounds responses in verifiable "
    "source material and reduces hallucinations significantly. The pipeline has "
    "three phases: indexing, retrieval, and generation. During indexing, documents "
    "are split into overlapping chunks, converted into embedding vectors, and stored "
    "in a vector database. During retrieval, the user query is embedded and compared "
    "to stored vectors using cosine similarity. Top-K chunks are returned. During "
    "generation, those chunks become the context injected into the LLM prompt, "
    "enabling it to answer with grounded, document-sourced information."
)

EMBEDDING_MODELS = (
    "Embedding models map text to dense numerical vectors capturing semantic meaning. "
    "Key models include Google's gemini-embedding-001 (3072-dim, MTEB leaderboard), "
    "OpenAI text-embedding-3-large, and open models like BAAI/bge-large-en-v1.5. "
    "Asymmetric retrieval is critical for RAG quality: use RETRIEVAL_DOCUMENT task "
    "type when embedding passages, and RETRIEVAL_QUERY when embedding questions. "
    "This tells the model to optimize vectors differently for storage vs. lookup. "
    "Matryoshka Representation Learning (MRL) allows truncating output dimensions "
    "at query time without full re-indexing, offering storage vs quality trade-offs."
)

# ── v1 only ───────────────────────────────────────────────────────────────────
VECTOR_DB_v1 = (
    "Vector databases store high-dimensional embedding vectors and support "
    "approximate nearest-neighbor (ANN) search. Popular options include ChromaDB, "
    "FAISS, and Pinecone. ChromaDB is ideal for local development — it requires no "
    "external server, persists data to disk, and supports metadata filtering. FAISS "
    "is Meta's open-source library optimized for in-memory similarity search. "
    "Pinecone is a managed cloud vector database offering serverless scaling. "
    "The primary indexing algorithm used across these systems is HNSW (Hierarchical "
    "Navigable Small World), providing sub-linear query time with high recall. "
    "Typical embedding dimensions range from 384 to 768 for most production RAG "
    "deployments, balancing storage cost against retrieval quality."
)

# ── v2 only (VECTOR_DB updated, LLM_INTEGRATION new) ─────────────────────────
VECTOR_DB_v2 = (
    "Vector databases store high-dimensional embedding vectors and support "
    "approximate nearest-neighbor (ANN) search. Leading options in 2025 include "
    "ChromaDB, FAISS, Pinecone, Weaviate, and Qdrant. Qdrant has emerged as a "
    "high-performance alternative with rich payload filtering and binary quantization "
    "support. Weaviate supports multi-modal vectors including text and images. "
    "The primary indexing algorithm remains HNSW (Hierarchical Navigable Small World). "
    "Advanced options include ScaNN from Google for production-grade throughput. "
    "Embedding dimensions have grown to 3072 with models like gemini-embedding-001. "
    "Matryoshka Representation Learning now allows dimension truncation at query time, "
    "which is a major operational improvement over older fixed-dimension models."
)

LLM_INTEGRATION_v2 = (
    "The generation phase uses an LLM to produce answers grounded in retrieved context. "
    "Key considerations: context window size determines how many chunks fit in the prompt, "
    "instruction following quality determines whether the model stays grounded, and "
    "temperature controls hallucination risk. For factual RAG, use temperature 0.1-0.2. "
    "Model options include GPT-4o, Gemini Flash, Llama 3.1-70B, and Mistral-Large. "
    "The prompt template is critical: always separate system instructions, retrieved "
    "context, and the user question clearly. Conversation history should be included "
    "in the prompt for multi-turn RAG sessions to enable follow-up questions."
)

# ── v3 only (LLM_INTEGRATION updated, RAG_EVALUATION new) ────────────────────
LLM_INTEGRATION_v3 = (
    "The generation phase uses an LLM to produce answers grounded in retrieved context. "
    "As of Q2 2025, leading models for RAG include GPT-4o, Gemini 2.0 Flash, "
    "Llama 3.1-70B, and gpt-oss-20b (OpenAI open-weight). Reasoning models show "
    "particular promise for multi-hop RAG where the model synthesizes across chunks. "
    "Re-ranking models (cross-encoders) are now standard in production stacks — they "
    "score query-chunk pairs directly, significantly improving precision before the "
    "generation step. Structured output (JSON mode) is increasingly used to enable "
    "downstream processing of answers in agent pipelines."
)

RAG_EVALUATION_v3 = (
    "Evaluating RAG pipelines requires measuring both retrieval and generation quality. "
    "Retrieval metrics: MRR (Mean Reciprocal Rank), NDCG, Recall@K, and Precision@K. "
    "Generation metrics: Faithfulness (is the answer grounded in context?), Answer "
    "Relevance, and Context Precision. The RAGAS framework automates evaluation using "
    "an LLM as judge — it computes faithfulness, answer relevance, context recall, "
    "and context precision in a single evaluation pass. Faithfulness is the most "
    "critical production metric: a fluent but hallucinated answer is worse than a "
    "less polished but fully grounded one. Always test with adversarial queries that "
    "have no answer in the documents — the model should say so rather than hallucinate."
)

# ════════════════════════════════════════════════════════════
# GENERATE PDFs
# ════════════════════════════════════════════════════════════

ACTIVE_PATH = "./pdfs/versions/rag_guide.pdf"   # ← this is what gets ingested
V2_STAGED   = "./pdfs/versions/rag_guide_v2_staged.pdf"
V3_STAGED   = "./pdfs/versions/rag_guide_v3_staged.pdf"

make_pdf(ACTIVE_PATH, "RAG Systems Guide — v1", {
    "1. Introduction to RAG":   INTRO,          # SHARED
    "2. Vector Databases":      VECTOR_DB_v1,   # v1 only
    "3. Embedding Models":      EMBEDDING_MODELS,  # SHARED
})

make_pdf(V2_STAGED, "RAG Systems Guide — v2", {
    "1. Introduction to RAG":   INTRO,            # UNCHANGED
    "2. Vector Databases":      VECTOR_DB_v2,     # CHANGED
    "3. Embedding Models":      EMBEDDING_MODELS, # UNCHANGED
    "4. LLM Integration":       LLM_INTEGRATION_v2,  # NEW
})

make_pdf(V3_STAGED, "RAG Systems Guide — v3", {
    "1. Introduction to RAG":   INTRO,              # UNCHANGED
    "2. Vector Databases":      VECTOR_DB_v2,       # UNCHANGED (from v2)
    "3. Embedding Models":      EMBEDDING_MODELS,   # UNCHANGED
    "4. LLM Integration":       LLM_INTEGRATION_v3, # CHANGED (from v2)
    "5. RAG Evaluation":        RAG_EVALUATION_v3,  # NEW
})

print("\n✅ Test PDFs created:")
print(f"   {ACTIVE_PATH}       ← v1 (active, ready to ingest)")
print(f"   {V2_STAGED}  ← v2 (staged, copy over active to trigger CDC)")
print(f"   {V3_STAGED}  ← v3 (staged, copy over active again)")
print()
print("Expected CDC behaviour:")
print("   v1 → v2 : Vector DB section changed, LLM section added")
print("   v2 → v3 : LLM section changed, Evaluation section added")
print("   Introduction + Embedding Models: NEVER re-embedded across all 3 versions")

✅ Created: ./pdfs/versions/rag_guide.pdf
✅ Created: ./pdfs/versions/rag_guide_v2_staged.pdf
✅ Created: ./pdfs/versions/rag_guide_v3_staged.pdf

✅ Test PDFs created:
   ./pdfs/versions/rag_guide.pdf       ← v1 (active, ready to ingest)
   ./pdfs/versions/rag_guide_v2_staged.pdf  ← v2 (staged, copy over active to trigger CDC)
   ./pdfs/versions/rag_guide_v3_staged.pdf  ← v3 (staged, copy over active again)

Expected CDC behaviour:
   v1 → v2 : Vector DB section changed, LLM section added
   v2 → v3 : LLM section changed, Evaluation section added
   Introduction + Embedding Models: NEVER re-embedded across all 3 versions


In [24]:
# ── Cell 14E: ▶️ Full Test Suite ──────────────────────────────────────────────
# Run top-to-bottom. Each section tests one feature.
# Total Gemini API calls are printed so you can verify CDC savings.

import shutil, time

ACTIVE_PATH = "./pdfs/versions/rag_guide.pdf"
V2_STAGED   = "./pdfs/versions/rag_guide_v2_staged.pdf"
V3_STAGED   = "./pdfs/versions/rag_guide_v3_staged.pdf"


# ════════════════════════════════════════════════════════════
# TEST 1: Full ingest of v1
# ════════════════════════════════════════════════════════════
print("\n" + "█"*60)
print("  TEST 1 — Full ingest of v1")
print("█"*60)

ingest_pdf_v2(ACTIVE_PATH)

print("\n📋 Staleness registry after v1 ingest:")
staleness_reg.show()
chunk_reg.show()


# ════════════════════════════════════════════════════════════
# TEST 2: Staleness check — same file, no changes
# ════════════════════════════════════════════════════════════
print("\n" + "█"*60)
print("  TEST 2 — Staleness check (unchanged file)")
print("█"*60)

ingest_pdf_v2(ACTIVE_PATH)   # should say UNCHANGED and skip


# ════════════════════════════════════════════════════════════
# TEST 3: Ask questions against v1
# ════════════════════════════════════════════════════════════
print("\n" + "█"*60)
print("  TEST 3 — Queries against v1")
print("█"*60)

ask_weighted("What vector databases does this guide mention?",
             show_scores=True)

ask_weighted("What is the recommended temperature for RAG?")


# ════════════════════════════════════════════════════════════
# TEST 4: Upgrade to v2 — CDC kicks in
# ════════════════════════════════════════════════════════════
print("\n" + "█"*60)
print("  TEST 4 — Upgrade to v2 (CDC update)")
print("█"*60)

print("📋 Copying v2 over active file...")
shutil.copy(V2_STAGED, ACTIVE_PATH)
time.sleep(1)   # ensure filesystem timestamp is flushed

ingest_pdf_v2(ACTIVE_PATH)
# Expected: Introduction + Embedding unchanged, Vector DB changed, LLM new

print("\n📋 Staleness registry after v2 update:")
staleness_reg.show()


# ════════════════════════════════════════════════════════════
# TEST 5: Same question — answer should now include v2 content
# ════════════════════════════════════════════════════════════
print("\n" + "█"*60)
print("  TEST 5 — Same queries after v2 CDC update")
print("█"*60)

print("(Recency scores: v2 chunks should score higher than v1 chunks)")

ask_weighted(
    "What vector databases does this guide mention?",
    show_scores=True,    # shows cosine vs recency breakdown
)

ask_weighted("What does the guide say about LLM integration?")


# ════════════════════════════════════════════════════════════
# TEST 6: Upgrade to v3 — second CDC cycle
# ════════════════════════════════════════════════════════════
print("\n" + "█"*60)
print("  TEST 6 — Upgrade to v3 (second CDC cycle)")
print("█"*60)

print("📋 Copying v3 over active file...")
shutil.copy(V3_STAGED, ACTIVE_PATH)
time.sleep(1)

ingest_pdf_v2(ACTIVE_PATH)
# Expected: Intro + Embedding + Vector DB unchanged, LLM changed, Evaluation new


# ════════════════════════════════════════════════════════════
# TEST 7: Recency verification — v3 chunks score highest
# ════════════════════════════════════════════════════════════
print("\n" + "█"*60)
print("  TEST 7 — Recency weighting verification")
print("█"*60)

print("\n🔬 Showing full score breakdown for new v3 content:")
ask_weighted(
    "How should RAG pipelines be evaluated?",
    show_scores=True,
)

print("\n🔬 Checking a shared section (should NOT have high recency boost):")
ask_weighted(
    "What is Retrieval-Augmented Generation?",
    show_scores=True,
)


# ════════════════════════════════════════════════════════════
# TEST 8: CDC + memory (full stack)
# ════════════════════════════════════════════════════════════
print("\n" + "█"*60)
print("  TEST 8 — Full stack: weighted retrieval + conversation memory")
print("█"*60)

test_memory = ConversationMemory()
ask_weighted("What is RAG and what databases does this guide recommend?",
             memory=test_memory)
ask_weighted("What metrics should I use to evaluate it?",
             memory=test_memory)   # follow-up — memory carries context
ask_weighted("How does that compare to what you said about the databases?",
             memory=test_memory)   # cross-turn reference


# ════════════════════════════════════════════════════════════
# SUMMARY
# ════════════════════════════════════════════════════════════
print("\n" + "█"*60)
print("  SUMMARY")
print("█"*60)

print("\n📋 Final staleness registry state:")
staleness_reg.show()

print("\n📊 Expected CDC efficiency:")
print("   v1→v2: Introduction + Embedding not re-embedded (shared content)")
print("   v2→v3: Introduction + Embedding + Vector DB not re-embedded")
print("   This demonstrates increasing CDC reuse across document versions.")

print("\n📊 Expected recency ordering (newest first):")
print("   v3 chunks (RAG Evaluation, updated LLM)      → highest recency score")
print("   v2 chunks (updated Vector DB, original LLM)  → medium recency score")
print("   v1 chunks (shared: Introduction, Embedding)  → lowest recency score")
print("   (All scores > 0 because half-life=30 days and test ran today)")


████████████████████████████████████████████████████████████
  TEST 1 — Full ingest of v1
████████████████████████████████████████████████████████████

📥 Smart Ingest: rag_guide.pdf
📌 Staleness status: NEW
📄 Step 1/3  Extracting text...
   ✅ 1 pages | 2,284 chars
✂️  Step 2/3  Chunking...
   ✅ 4 chunks
🧠 Step 3/3  Embedding via gemini-embedding-001...
  Embedding chunk 4/4...
  ✅ Embedded 4 chunks

✅ Full ingest complete!
   4 chunks stored
   Hash       : 6ea2f4cb68218e4876c586c7...
   Ingested at: 2026-04-13T10:55:51

📋 Staleness registry after v1 ingest:

📋 Staleness Registry (1 docs)
────────────────────────────────────────────────────────────
  ✅ rag_guide.pdf
     Version    : v1
     Ingested   : 2026-04-13 10:55:55
     Chunks     : 4
     Hash       : 6ea2f4cb68218e4876c586c7...


🗂️  Chunk Registry (1 docs):
   rag_guide.pdf: 4 chunks tracked

████████████████████████████████████████████████████████████
  TEST 2 — Staleness check (unchanged file)
████████████████████████████